#### Company Brochure Generator using Gradio

- For website contents extraction, we have used gemini-3.6-flash irrespective of the model selected from the end user.

- Model selected by the end user has been used just to create the brochure from the extracted contents [summarize]

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

GROQ_BASE_URL = os.getenv("GROQ_BASE_URL")
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)
gpt = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)

In [2]:
from scraper_for_013 import fetch_website_contents, fetch_website_links
import json

In [3]:
system_prompt_to_select_relevant_links = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_user_prompt_to_select_relevant_links(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += '\n'.join(links)
    return user_prompt

In [5]:
def select_relevant_links(url):
    print(f'Selecting the relevant links for {url} using GEMINI-3.6-flash')
    response = gemini.chat.completions.create(
        model = 'gemini-3.6-flash',
        messages=[
            {'role':'system', 'content':system_prompt_to_select_relevant_links},
            {'role':'user', 'content':get_user_prompt_to_select_relevant_links(url)}
        ],
        response_format={'type':'json_object'}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [6]:
def fetch_website_contents_for_landing_page_and_relevant_links(url):
    landing_page_contents = fetch_website_links(url)
    relevant_links = select_relevant_links(url)
    contents = f"## Landing Page contents: \n\n {landing_page_contents}\n## Relevant Links: \n"
    for link in relevant_links['links']:
        contents += f"\n\n### Link : {link['type']}\n"
        contents += fetch_website_contents(link['url'])
    return contents

In [7]:
system_prompt_to_generate_brochure = '''
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
'''

In [8]:
def get_user_prompt_to_generate_brochure(company_name, url):
        user_prompt = f'''
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
'''
        user_prompt += fetch_website_contents_for_landing_page_and_relevant_links(url)
        user_prompt = user_prompt[0:5000]
        return user_prompt

In [9]:
def create_brochure(company_name, url, model):
    try:
        messages = [
            {'role': 'system', 'content': system_prompt_to_generate_brochure},
            {'role': 'user', 'content': get_user_prompt_to_generate_brochure(company_name, url)}
        ]

        if model == 'GEMINI':
            stream = gemini.chat.completions.create(
                model='gemini-3.6-flash',
                messages=messages,
                stream=True
            )
        elif model == 'GPT':
            stream = gpt.chat.completions.create(
                model='openai/gpt-oss-120b',
                messages=messages,
                stream=True
            )
        else:
            raise ValueError('Unknown model')

        result = ""
        for chunk in stream:
            result += chunk.choices[0].delta.content or ""
            yield result

    except Exception as e:
        yield f"Error: {e}"

In [10]:
import gradio as gr

/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
name_input = gr.Textbox(label='Company Name: ')
url_input = gr.Textbox(label="Landing page URL [including http:// or https://]")
model_selector = gr.Dropdown(['GEMINI', 'GPT'], label='Select the Model', value='GPT')
brochure_output = gr.Markdown('Company Brochure: ')

view = gr.Interface(
    fn=create_brochure,
    inputs = [name_input, url_input, model_selector],
    outputs = [brochure_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "GEMINI"]
    ],
    flagging_mode = "never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


Selecting the relevant links for https://groq.com using GEMINI-3.6-flash
Found 9 relevant links
Selecting the relevant links for https://groq.com using GEMINI-3.6-flash
Found 7 relevant links
